# MELD 7-Class Emotion Classification — UFEN + MTFN with BERT

**Dataset upload instructions:**
1. Create a Kaggle dataset with your MELD files:
   - `train.pkl`, `dev.pkl`, `test.pkl`, `embedding.p`
   - `raw/train_sent_emo.csv`, `raw/dev_sent_emo.csv`, `raw/test_sent_emo.csv`
2. Add that dataset to this notebook via **Add Data**
3. Update `DATA_DIR` and `RAW_CSV_DIR` in the Config cell below

**GPU:** Enable GPU in Settings → Accelerator (T4 x2 or P100 recommended)

In [ ]:
!pip install -q transformers tqdm

In [ ]:
import os, sys, csv, math, random, pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import GradScaler, autocast
from torch.optim import Adam
from torch.optim.lr_scheduler import LambdaLR
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from transformers import BertModel, BertTokenizer
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from tqdm.notebook import tqdm
from types import SimpleNamespace

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GiB')

## Config
Change `MODALITIES` to run ablations:
- `['text', 'audio', 'video']` — full 3-modal (Exp 6)
- `['text', 'audio']` — bimodal ablation (Exp 7a)
- `['text', 'video']` — bimodal ablation (Exp 7b)
- `['text']` — text-only unimodal (Exp 7c)

In [ ]:
# ---- Path to your uploaded MELD dataset ----
DATA_DIR    = '/kaggle/input/meld-dataset/meld'          # update to your dataset path
RAW_CSV_DIR = '/kaggle/input/meld-dataset/meld/raw'      # update to your dataset path

# ---- Modalities (change for ablation) ----
MODALITIES = ['text', 'audio', 'video']   # full 3-modal

config = SimpleNamespace(
    data_dir        = DATA_DIR,
    raw_csv_dir     = RAW_CSV_DIR,
    embedding_path  = f'{DATA_DIR}/embedding.p',
    batch_size      = 32,           # can go higher on Kaggle GPUs
    modalities      = MODALITIES,

    # model dimensions
    d_m             = 128,
    conv_dim        = 64,
    n_layers        = 2,
    kernel_sizes    = [1, 5],
    d_ff            = 128,
    num_classes     = 7,

    # feature dimensions
    text_size       = 300,
    visual_size     = 2048,
    visual_proj_dim = None,         # no projection — Kaggle GPU has enough VRAM
    acoustic_size   = 32,

    # text encoder
    use_bert        = True,

    # attention
    self_att_heads  = 1,
    cross_att_heads = 4,
    att_dropout     = 0.2,
    dropout         = 0.1,

    # training
    lr              = 1e-3,
    lr_bert         = 2e-5,
    epochs          = 50,
    early_stop      = 10,
    grad_clip       = 1.0,
    use_lr_scheduler  = True,
    use_bert_warmup   = True,
    bert_warmup_epochs= 3,
    label_smoothing = 0.1,

    # loss
    use_focal_loss  = False,
    focal_gamma     = 2.0,
    use_class_weights = True,

    pretrained_emb  = None,
)

EMOTION_LABELS = ['Neutral', 'Surprise', 'Fear', 'Sadness', 'Joy', 'Disgust', 'Anger']
NUM_CLASSES    = 7
print(f'Modalities: {config.modalities}')

## Dataset & DataLoader

In [ ]:
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

CSV_FILES = {
    'train': 'train_sent_emo.csv',
    'dev':   'dev_sent_emo.csv',
    'test':  'test_sent_emo.csv',
}

def _load_csv_text(csv_path):
    lookup = {}
    with open(csv_path) as f:
        for row in csv.DictReader(f):
            key = f"{row['Dialogue_ID']}_{row['Utterance_ID']}"
            lookup[key] = row['Utterance']
    return lookup


class MELDDataset(Dataset):
    def __init__(self, pkl_path, csv_path):
        with open(pkl_path, 'rb') as f:
            raw = pickle.load(f)
        text_lookup = _load_csv_text(csv_path)
        self.data = []
        for k, sample in raw.items():
            sample['text'] = text_lookup.get(k, '')
            self.data.append(sample)

    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i]


def collate_fn(batch):
    batch = sorted(batch, key=lambda x: len(x['token_ids']), reverse=True)
    lengths   = torch.LongTensor([len(s['token_ids']) for s in batch])
    labels    = torch.LongTensor([s['label'] for s in batch])
    token_ids = pad_sequence(
        [torch.LongTensor([int(x) for x in s['token_ids']]) for s in batch],
        batch_first=True, padding_value=0)
    audio = pad_sequence(
        [torch.FloatTensor(s['audio_features']) for s in batch],
        batch_first=True, padding_value=0.0)
    video = pad_sequence(
        [torch.FloatTensor(np.asarray(s['video_features'], dtype=np.float32)) for s in batch],
        batch_first=True, padding_value=0.0)
    bert_details = [bert_tokenizer(
        s['text'], max_length=64, add_special_tokens=True,
        padding='max_length', truncation=True) for s in batch]
    bert_ids      = torch.LongTensor([d['input_ids']      for d in bert_details])
    bert_mask     = torch.LongTensor([d['attention_mask'] for d in bert_details])
    bert_type_ids = torch.LongTensor([d['token_type_ids'] for d in bert_details])
    return token_ids, audio, video, labels, lengths, bert_ids, bert_mask, bert_type_ids


def get_loader(cfg, split, shuffle=True, generator=None, worker_init_fn=None):
    pkl_path = f'{cfg.data_dir}/{split}.pkl'
    csv_path = f'{cfg.raw_csv_dir}/{CSV_FILES[split]}'
    return DataLoader(
        MELDDataset(pkl_path, csv_path),
        batch_size=cfg.batch_size, shuffle=shuffle,
        collate_fn=collate_fn, generator=generator,
        worker_init_fn=worker_init_fn,
    )

## Model (UFEN + MTFN)

In [ ]:
def masked_mean(x, mask=None):
    if mask is None:
        return x.mean(dim=1)
    valid = (~mask).float().unsqueeze(-1)
    return (x * valid).sum(dim=1) / valid.sum(dim=1).clamp(min=1.0)


class UFEN(nn.Module):
    def __init__(self, input_dim, d_m, num_classes, n_layers=2, kernel_sizes=None,
                 conv_dim=64, n_att_heads=1, dropout=0.1):
        super().__init__()
        kernel_sizes = kernel_sizes or [1, 3]
        assert len(kernel_sizes) == n_layers
        self.bigru     = nn.GRU(input_dim, d_m // 2, batch_first=True, bidirectional=True)
        self.convs     = nn.ModuleList([nn.Conv1d(d_m, conv_dim, k, padding=k//2) for k in kernel_sizes])
        self.self_atts = nn.ModuleList([nn.MultiheadAttention(conv_dim, n_att_heads, dropout=dropout, batch_first=True) for _ in range(n_layers)])
        self.unpools   = nn.ModuleList([nn.Linear(conv_dim, d_m) for _ in range(n_layers)])
        self.layer_norm = nn.LayerNorm(d_m)
        self.dropout    = nn.Dropout(dropout)
        self.pred_head  = nn.Linear(d_m, num_classes)

    def forward(self, x, mask=None):
        x, _ = self.bigru(x)
        if mask is not None:
            x = x.masked_fill(mask.unsqueeze(-1), 0.0)
        branch_outputs = []
        for conv, self_att, unpool in zip(self.convs, self.self_atts, self.unpools):
            c = F.relu(conv(x.transpose(1, 2)).transpose(1, 2))
            if mask is not None:
                c = c.masked_fill(mask.unsqueeze(-1), 0.0)
            att_out, _ = self_att(c, c, c, key_padding_mask=mask)
            branch_outputs.append(unpool(c * att_out))
        fusion = self.dropout(self.layer_norm(sum(branch_outputs)))
        if mask is not None:
            fusion = fusion.masked_fill(mask.unsqueeze(-1), 0.0)
        return fusion, self.pred_head(masked_mean(fusion, mask))


class CrossModalAttention(nn.Module):
    def __init__(self, d_m, n_heads, att_dropout=0.2):
        super().__init__()
        self.mha = nn.MultiheadAttention(d_m, n_heads, dropout=att_dropout, batch_first=True)
        self.layer_norm = nn.LayerNorm(d_m)
    def forward(self, x_i, x_j, mask_i=None, mask_j=None):
        out, _ = self.mha(x_i, x_j, x_j, key_padding_mask=mask_j)
        return self.layer_norm(out)


class EncoderBlock(nn.Module):
    def __init__(self, d_m, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_m, n_heads, dropout=dropout, batch_first=True)
        self.ff        = nn.Sequential(nn.Linear(d_m, d_ff), nn.ReLU(), nn.Linear(d_ff, d_m))
        self.norm1, self.norm2 = nn.LayerNorm(d_m), nn.LayerNorm(d_m)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        attn, _ = self.self_attn(x, x, x)
        x = self.norm1(x + self.drop(attn))
        return self.norm2(x + self.drop(self.ff(x)))


class DecoderBlock(nn.Module):
    def __init__(self, d_m, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(d_m, n_heads, dropout=dropout, batch_first=True)
        self.ff         = nn.Sequential(nn.Linear(d_m, d_ff), nn.ReLU(), nn.Linear(d_ff, d_m))
        self.norm1, self.norm2 = nn.LayerNorm(d_m), nn.LayerNorm(d_m)
        self.drop = nn.Dropout(dropout)
    def forward(self, x, memory):
        attn, _ = self.cross_attn(x, memory, memory)
        x = self.norm1(x + self.drop(attn))
        return self.norm2(x + self.drop(self.ff(x)))


class MTFN(nn.Module):
    def __init__(self, d_m, num_classes, modalities, n_cross_heads=4, d_ff=256,
                 dropout=0.1, att_dropout=0.2):
        super().__init__()
        self.modalities = modalities
        self.projs = nn.ModuleDict({m: nn.Linear(d_m, d_m) for m in modalities})
        self.cross_atts = nn.ModuleDict({
            f'{mi}2{mj}': CrossModalAttention(d_m, n_cross_heads, att_dropout)
            for mi in modalities for mj in modalities if mi != mj
        })
        n_pairs = len(modalities) * (len(modalities) - 1)
        self.encoder     = EncoderBlock(d_m, n_cross_heads, d_ff, dropout)
        self.decoder     = DecoderBlock(d_m, n_cross_heads, d_ff, dropout)
        self.pred_fusion = nn.Linear(n_pairs * d_m, num_classes)
        self.pred_recon  = nn.Linear(d_m, num_classes)

    def forward(self, feats, masks):
        proj   = {m: self.projs[m](feats[m]) for m in self.modalities}
        tokens = []
        for mi in self.modalities:
            for mj in self.modalities:
                if mi != mj:
                    ca = self.cross_atts[f'{mi}2{mj}'](proj[mi], proj[mj], masks.get(mi), masks.get(mj))
                    tokens.append(masked_mean(ca, masks.get(mi)))
        tokens    = torch.stack(tokens, dim=1)
        y_m       = self.pred_fusion(tokens.reshape(tokens.size(0), -1))
        enc_out   = self.encoder(tokens)
        dec_out   = self.decoder(tokens, enc_out)
        y_m_prime = self.pred_recon(dec_out.mean(dim=1))
        return y_m, y_m_prime


class MultiTaskModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.use_bert   = getattr(cfg, 'use_bert', False)
        self.modalities = getattr(cfg, 'modalities', ['text', 'audio', 'video'])
        C = cfg.num_classes

        if 'text' in self.modalities:
            if self.use_bert:
                self.bert = BertModel.from_pretrained('bert-base-uncased')
                t_dim = 768
            else:
                vs, ed = cfg.pretrained_emb.shape
                self.text_embedding = nn.Embedding(vs, ed, padding_idx=0)
                self.text_embedding.weight.data.copy_(torch.from_numpy(cfg.pretrained_emb))
                self.text_embedding.weight.requires_grad = True
                t_dim = ed
            self.norm_t = nn.LayerNorm(t_dim)
            self.ufen_t = UFEN(t_dim, cfg.d_m, C, cfg.n_layers, cfg.kernel_sizes,
                               cfg.conv_dim, cfg.self_att_heads, cfg.dropout)

        if 'video' in self.modalities:
            vp = getattr(cfg, 'visual_proj_dim', None)
            self.video_proj = nn.Linear(cfg.visual_size, vp) if vp else None
            v_dim = vp or cfg.visual_size
            self.norm_v = nn.LayerNorm(v_dim)
            self.ufen_v = UFEN(v_dim, cfg.d_m, C, cfg.n_layers, cfg.kernel_sizes,
                               cfg.conv_dim, cfg.self_att_heads, cfg.dropout)

        if 'audio' in self.modalities:
            self.norm_a = nn.LayerNorm(cfg.acoustic_size)
            self.ufen_a = UFEN(cfg.acoustic_size, cfg.d_m, C, cfg.n_layers, cfg.kernel_sizes,
                               cfg.conv_dim, cfg.self_att_heads, cfg.dropout)

        self.mtfn = MTFN(cfg.d_m, C, self.modalities, cfg.cross_att_heads, cfg.d_ff,
                         cfg.dropout, cfg.att_dropout) if len(self.modalities) >= 2 else None

    def forward(self, token_ids, audio, video, av_mask,
                bert_ids=None, bert_mask=None, bert_type_ids=None):
        feats, masks, preds = {}, {}, {}

        if 'text' in self.modalities:
            if self.use_bert:
                text_emb = self.norm_t(self.bert(bert_ids, bert_mask, bert_type_ids).last_hidden_state)
                t_mask   = (bert_mask == 0)
            else:
                text_emb = self.norm_t(self.text_embedding(token_ids))
                t_mask   = av_mask
            feat_t, y_t = self.ufen_t(text_emb, t_mask)
            feats['text'], masks['text'], preds['text'] = feat_t, t_mask, y_t

        if 'video' in self.modalities:
            v = self.video_proj(video) if self.video_proj else video
            feat_v, y_v = self.ufen_v(self.norm_v(v), av_mask)
            feats['video'], masks['video'], preds['video'] = feat_v, av_mask, y_v

        if 'audio' in self.modalities:
            feat_a, y_a = self.ufen_a(self.norm_a(audio), av_mask)
            feats['audio'], masks['audio'], preds['audio'] = feat_a, av_mask, y_a

        if self.mtfn:
            y_m, y_m_prime = self.mtfn(feats, masks)
            preds['fusion'], preds['recon'] = y_m, y_m_prime
        else:
            preds['fusion'] = preds['recon'] = preds[self.modalities[0]]

        return preds

## Training Utilities

In [ ]:
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def _worker_init_fn(worker_id):
    s = torch.initial_seed() % (2**32)
    np.random.seed(s); random.seed(s)

def make_padding_mask(lengths, max_len):
    return torch.arange(max_len, device=lengths.device).unsqueeze(0) >= lengths.unsqueeze(1)

def unpack_batch(batch, dev, use_bert=False):
    token_ids, audio, video, labels, lengths = [x.to(dev) if hasattr(x,'to') else x for x in batch[:5]]
    token_ids = token_ids.long(); audio = audio.float(); video = video.float()
    labels = labels.long(); lengths = lengths.long()
    av_mask = make_padding_mask(lengths, token_ids.size(1))
    r = dict(token_ids=token_ids, audio=audio, video=video, labels=labels, av_mask=av_mask)
    if use_bert:
        r['bert_ids']      = batch[5].long().to(dev)
        r['bert_mask']     = batch[6].long().to(dev)
        r['bert_type_ids'] = batch[7].long().to(dev)
    return r

def compute_metrics(preds, labels):
    return {
        'Accuracy':    accuracy_score(labels, preds) * 100,
        'F1_weighted': f1_score(labels, preds, average='weighted', zero_division=0) * 100,
        'F1_macro':    f1_score(labels, preds, average='macro',    zero_division=0) * 100,
    }

def compute_class_weights(loader):
    counts = np.zeros(NUM_CLASSES)
    for batch in loader:
        for l in batch[3].numpy(): counts[l] += 1
    w = 1.0 / np.clip(counts, 1, None)
    return torch.FloatTensor(w / w.mean())

def evaluate(model, loader, use_bert=False):
    model.eval()
    use_amp = torch.cuda.is_available()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            b = unpack_batch(batch, device, use_bert)
            with autocast('cuda', enabled=use_amp):
                out = model(b['token_ids'], b['audio'], b['video'], b['av_mask'],
                            b.get('bert_ids'), b.get('bert_mask'), b.get('bert_type_ids'))
            all_preds.append(out['recon'].argmax(1).cpu().numpy())
            all_labels.append(b['labels'].cpu().numpy())
    preds  = np.concatenate(all_preds)
    labels = np.concatenate(all_labels)
    return compute_metrics(preds, labels), preds, labels

def print_full_report(preds, labels):
    print('\n--- Classification Report ---')
    print(classification_report(labels, preds, target_names=EMOTION_LABELS, zero_division=0))
    print('--- Confusion Matrix ---')
    cm = confusion_matrix(labels, preds)
    print('          ' + ' '.join(f'{EMOTION_LABELS[i][:4]:>5}' for i in range(NUM_CLASSES)))
    for i in range(NUM_CLASSES):
        print(f"{EMOTION_LABELS[i]:<10}" + ' '.join(f'{cm[i,j]:5d}' for j in range(NUM_CLASSES)))

class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0, label_smoothing=0.0):
        super().__init__()
        self.gamma, self.weight, self.ls = gamma, weight, label_smoothing
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight, label_smoothing=self.ls, reduction='none')
        return (((1 - torch.exp(-ce)) ** self.gamma) * ce).mean()

## Training

In [ ]:
set_seed(42)
use_bert   = config.use_bert
modalities = config.modalities

g = torch.Generator(); g.manual_seed(42)
train_loader = get_loader(config, 'train', shuffle=True, generator=g, worker_init_fn=_worker_init_fn)
dev_loader   = get_loader(config, 'dev',   shuffle=False)
test_loader  = get_loader(config, 'test',  shuffle=False)
print(f'Train={len(train_loader.dataset)}, Dev={len(dev_loader.dataset)}, Test={len(test_loader.dataset)}')

class_weights = compute_class_weights(train_loader).to(device) if config.use_class_weights else None
print(f'Class weights: {class_weights.cpu().numpy().round(3)}')

model    = MultiTaskModel(config).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {n_params:,}')

# Optimizer
if use_bert and 'text' in modalities:
    bert_ids_set = set(id(p) for p in model.bert.parameters())
    optimizer = Adam([
        {'params': model.bert.parameters(),                                            'lr': config.lr_bert},
        {'params': [p for p in model.parameters() if id(p) not in bert_ids_set], 'lr': config.lr},
    ])
    warmup = config.bert_warmup_epochs if config.use_bert_warmup else 0
    bert_lam  = lambda t: ((t+1)/warmup if t < warmup else max(0., .5*(1+math.cos(math.pi*(t-warmup)/max(1,config.epochs-warmup))))) if config.use_bert_warmup else (max(0.,.5*(1+math.cos(math.pi*t/max(1,config.epochs)))) if config.use_lr_scheduler else 1.)
    other_lam = lambda t: max(0., .5*(1+math.cos(math.pi*t/max(1,config.epochs)))) if config.use_lr_scheduler else 1.
    scheduler = LambdaLR(optimizer, [bert_lam, other_lam])
    print(f'BERT lr={config.lr_bert}, other lr={config.lr}, warmup={warmup} epochs')
else:
    optimizer = Adam(model.parameters(), lr=config.lr)
    scheduler = LambdaLR(optimizer, lambda t: max(0.,.5*(1+math.cos(math.pi*t/max(1,config.epochs))))) if config.use_lr_scheduler else None

ce_loss = FocalLoss(class_weights, config.focal_gamma, config.label_smoothing) \
          if config.use_focal_loss else \
          nn.CrossEntropyLoss(weight=class_weights, label_smoothing=config.label_smoothing)
print(f'Loss: {"Focal" if config.use_focal_loss else "CrossEntropy"}')

In [ ]:
best_dev_f1, best_epoch = 0.0, 0
os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
use_amp = torch.cuda.is_available()
scaler  = GradScaler('cuda', enabled=use_amp)

for epoch in range(1, config.epochs + 1):
    model.train()
    total_loss = 0.0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch:02d}', leave=True)

    for batch in pbar:
        b = unpack_batch(batch, device, use_bert)
        with autocast('cuda', enabled=use_amp):
            out  = model(b['token_ids'], b['audio'], b['video'], b['av_mask'],
                         b.get('bert_ids'), b.get('bert_mask'), b.get('bert_type_ids'))
            loss = sum(ce_loss(out[k], b['labels']) for k in list(modalities) + ['fusion', 'recon'])
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
        scaler.step(optimizer); scaler.update()
        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    if scheduler: scheduler.step()
    avg_loss = total_loss / len(train_loader)
    dev_metrics, _, _ = evaluate(model, dev_loader, use_bert)
    print(f"Epoch {epoch:02d} | loss={avg_loss:.4f} | Acc={dev_metrics['Accuracy']:.1f} | F1w={dev_metrics['F1_weighted']:.1f} | F1m={dev_metrics['F1_macro']:.1f}")

    if dev_metrics['F1_weighted'] > best_dev_f1:
        best_dev_f1, best_epoch = dev_metrics['F1_weighted'], epoch
        torch.save(model.state_dict(), '/kaggle/working/checkpoints/best_model.pt')
        print(f'  -> New best Dev F1w={best_dev_f1:.2f} — saved.')
    elif (epoch - best_epoch) >= config.early_stop:
        print(f'Early stopping at epoch {epoch} (best: {best_epoch})')
        break

print(f'\nLoading best checkpoint (epoch {best_epoch})...')
model.load_state_dict(torch.load('/kaggle/working/checkpoints/best_model.pt', map_location=device))
test_metrics, test_preds, test_labels = evaluate(model, test_loader, use_bert)

print('\n========== Test Results ==========')
print(f"Accuracy       : {test_metrics['Accuracy']:.1f}")
print(f"F1 (weighted)  : {test_metrics['F1_weighted']:.1f}")
print(f"F1 (macro)     : {test_metrics['F1_macro']:.1f}")
print('==================================')
print_full_report(test_preds, test_labels)